In [ ]:
from pathlib import Path
import json
import os
import re
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 12
MIN_IMAGES_PER_CLASS = 100
MAX_CLASSES = 20
LABEL_COLUMN = "subCategory"

PROJECT_DIR = Path.cwd()
DATASET_CSV = None
IMAGE_DIR = None

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("Project folder:", PROJECT_DIR)
print("Available devices:", tf.config.list_physical_devices())

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

fashion_classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

In [ ]:
fashion_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation="softmax")
])

fashion_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fashion_history = fashion_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

fashion_test_loss, fashion_test_accuracy = fashion_model.evaluate(x_test, y_test, verbose=0)
print(f"Fashion-MNIST test accuracy: {fashion_test_accuracy:.4f}")

In [ ]:
def find_catalog_files():
    search_roots = [PROJECT_DIR, *list(PROJECT_DIR.parents)[:4]]
    dataset_folders = [
        "fashion-dataset",
        "fashion_product_images",
        "fashion-product-images",
        "fashion product images"
    ]

    csv_candidates = []
    if DATASET_CSV is not None:
        csv_candidates.append(Path(DATASET_CSV))

    for root in search_roots:
        csv_candidates.extend([
            root / "styles.csv",
            root / "data" / "styles.csv"
        ])
        for folder in dataset_folders:
            csv_candidates.extend([
                root / folder / "styles.csv",
                root / "data" / folder / "styles.csv"
            ])

    csv_path = next((path for path in csv_candidates if path.is_file()), None)

    if csv_path is None:
        raise FileNotFoundError(
            "styles.csv was not found. Set DATASET_CSV to the full path of styles.csv in the first cell."
        )

    image_candidates = []
    if IMAGE_DIR is not None:
        image_candidates.append(Path(IMAGE_DIR))

    image_candidates.extend([
        csv_path.parent / "images",
        csv_path.parent / "images_compressed",
        csv_path.parent / "image",
        csv_path.parent.parent / "images",
        csv_path.parent.parent / "images_compressed"
    ])

    image_dir = next((path for path in image_candidates if path.is_dir()), None)

    if image_dir is None:
        raise FileNotFoundError(
            "The product images folder was not found. Set IMAGE_DIR to the folder containing the .jpg files in the first cell."
        )

    return csv_path, image_dir


DATASET_CSV, IMAGE_DIR = find_catalog_files()
print("Catalog CSV:", DATASET_CSV)
print("Images folder:", IMAGE_DIR)

In [ ]:
styles = pd.read_csv(DATASET_CSV, dtype={"id": "string"}, on_bad_lines="skip")
styles.columns = styles.columns.str.strip()

required_columns = {"id", LABEL_COLUMN}
missing_columns = required_columns.difference(styles.columns)

if missing_columns:
    raise ValueError(f"Missing required columns in styles.csv: {sorted(missing_columns)}")

catalog = styles.dropna(subset=["id", LABEL_COLUMN]).copy()
catalog["id"] = catalog["id"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
catalog[LABEL_COLUMN] = catalog[LABEL_COLUMN].astype(str).str.strip()
catalog = catalog[(catalog["id"] != "") & (catalog[LABEL_COLUMN] != "")]
catalog = catalog.drop_duplicates(subset="id")
catalog["image_path"] = catalog["id"].map(lambda image_id: str(IMAGE_DIR / f"{image_id}.jpg"))
catalog = catalog[catalog["image_path"].map(os.path.isfile)].copy()

category_counts = catalog[LABEL_COLUMN].value_counts()
selected_categories = category_counts[category_counts >= MIN_IMAGES_PER_CLASS].head(MAX_CLASSES).index
catalog = catalog[catalog[LABEL_COLUMN].isin(selected_categories)].copy()

if catalog[LABEL_COLUMN].nunique() < 2:
    raise ValueError(
        "Not enough product categories were found. Check the image folder and lower MIN_IMAGES_PER_CLASS if needed."
    )

class_names = sorted(catalog[LABEL_COLUMN].unique().tolist())
class_to_index = {name: index for index, name in enumerate(class_names)}
catalog["label_id"] = catalog[LABEL_COLUMN].map(class_to_index).astype("int32")
catalog = catalog.reset_index(drop=True)

print("Usable product images:", len(catalog))
print("Number of categories:", len(class_names))
print("Categories:", class_names)
print(catalog[LABEL_COLUMN].value_counts())

In [ ]:
train_df, test_df = train_test_split(
    catalog,
    test_size=0.2,
    random_state=SEED,
    stratify=catalog["label_id"]
)

train_df, validation_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["label_id"]
)


def load_product_image(image_path, label):
    image_content = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image_content, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image.set_shape((IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    return image, tf.cast(label, tf.int32)


def create_image_dataset(frame, training=False):
    image_paths = frame["image_path"].astype(str).to_numpy()
    labels = frame["label_id"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(load_product_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return dataset


train_dataset = create_image_dataset(train_df, training=True)
validation_dataset = create_image_dataset(validation_df)
test_dataset = create_image_dataset(test_df)

print("Training images:", len(train_df))
print("Validation images:", len(validation_df))
print("Test images:", len(test_df))

In [ ]:
class_values = np.sort(train_df["label_id"].unique())
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_values,
    y=train_df["label_id"].to_numpy()
)
class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(class_values, class_weights_array)
}

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1)
])

catalog_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    augmentation,
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(class_names), activation="softmax")
])

catalog_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

catalog_model.summary()

In [ ]:
MODEL_DIR = PROJECT_DIR / "models" / "cv"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "retailmind_catalog_classifier.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

catalog_history = catalog_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
test_results = catalog_model.evaluate(test_dataset, verbose=1, return_dict=True)
test_loss = float(test_results["loss"])
test_accuracy = float(test_results["accuracy"])

true_labels = np.concatenate([labels.numpy() for _, labels in test_dataset])
predicted_probabilities = catalog_model.predict(test_dataset, verbose=1)
predicted_labels = np.argmax(predicted_probabilities, axis=1)

catalog_accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Catalog classifier test accuracy: {catalog_accuracy:.4f}")
print(f"Catalog classifier test loss: {test_loss:.4f}")

In [ ]:
class_ids = np.arange(len(class_names))
confusion = confusion_matrix(true_labels, predicted_labels, labels=class_ids)

display = ConfusionMatrixDisplay(
    confusion_matrix=confusion,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(14, 12))
display.plot(
    ax=ax,
    include_values=False,
    xticks_rotation="vertical",
    cmap="Blues",
    colorbar=True
)
ax.set_title("RetailMind AI - Product Category Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(
    true_labels,
    predicted_labels,
    labels=class_ids,
    target_names=class_names,
    zero_division=0
))

In [ ]:
catalog_model.save(MODEL_PATH)

CLASS_NAMES_PATH = MODEL_DIR / "retailmind_catalog_classes.json"
METRICS_PATH = MODEL_DIR / "retailmind_catalog_metrics.json"

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=4)

metrics = {
    "project": "RetailMind AI",
    "module": "Day 6 - CNN Catalog Classifier",
    "label_column": LABEL_COLUMN,
    "image_size": list(IMAGE_SIZE),
    "number_of_classes": len(class_names),
    "class_names": class_names,
    "training_images": int(len(train_df)),
    "validation_images": int(len(validation_df)),
    "test_images": int(len(test_df)),
    "fashion_mnist_test_accuracy": float(fashion_test_accuracy),
    "catalog_test_loss": test_loss,
    "catalog_test_accuracy": float(catalog_accuracy)
}

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Model saved to:", MODEL_PATH)
print("Class names saved to:", CLASS_NAMES_PATH)
print("Metrics saved to:", METRICS_PATH)

In [ ]:
README_PATH = PROJECT_DIR / "README.md"
README_HEADING = "## Day 6 - CNN Catalog Classifier"
readme_section = (
    f"{README_HEADING}\n\n"
    f"- Fashion-MNIST prototype test accuracy: {fashion_test_accuracy:.2%}\n"
    f"- Product catalog test accuracy: {catalog_accuracy:.2%}\n"
    f"- Product categories: {len(class_names)}\n"
    f"- Label column: `{LABEL_COLUMN}`\n"
    f"- Saved model: `{MODEL_PATH.relative_to(PROJECT_DIR)}`\n"
)

if README_PATH.exists():
    readme_text = README_PATH.read_text(encoding="utf-8")
else:
    readme_text = "# RetailMind AI\n"

section_pattern = re.compile(
    r"(?ms)^## Day 6 - CNN Catalog Classifier\n.*?(?=^## |\Z)"
)

if section_pattern.search(readme_text):
    readme_text = section_pattern.sub(readme_section, readme_text).rstrip() + "\n"
else:
    readme_text = readme_text.rstrip() + "\n\n" + readme_section

README_PATH.write_text(readme_text, encoding="utf-8")
print("README updated:", README_PATH)
print(readme_section)

In [ ]:
print("Day 6 completed")
print(f"Fashion-MNIST accuracy: {fashion_test_accuracy:.2%}")
print(f"Product catalog accuracy: {catalog_accuracy:.2%}")
print("Saved CV module:", MODEL_PATH)
print("README:", README_PATH)